In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import time

spark = SparkSession.builder \
    .appName("salting_demo") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df = spark.read.parquet("/home/aki/Documents/ad-engine/data/bronze")
df.cache()
df.count()  # materialize cache

print("Total rows:", df.count())
print("\nCampaign distribution (top 5):")
df.groupBy("campaign_id").count() \
  .orderBy(F.col("count").desc()) \
  .show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 12:25:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Total rows: 10000000

Campaign distribution (top 5):


+-----------+-------+
|campaign_id|  count|
+-----------+-------+
|   camp_001|3168697|
|   camp_003|3165763|
|   camp_002|3164740|
|   camp_974|    580|
|   camp_716|    574|
+-----------+-------+
only showing top 5 rows


In [3]:
import os

output_path = "/home/aki/Documents/ad-engine/data/silver"
os.makedirs(output_path, exist_ok=True)

In [4]:
df.withColumn("partition", F.spark_partition_id()).groupBy("partition").count().orderBy("partition").show()

+---------+------+
|partition| count|
+---------+------+
|        0|538186|
|        1|538191|
|        2|538184|
|        3|538183|
|        4|538185|
|        5|538182|
|        6|538185|
|        7|538182|
|        8|538187|
|        9|538180|
|       10|538188|
|       11|538185|
|       12|538178|
|       13|538184|
|       14|538183|
|       15|538192|
|       16|538175|
|       17|850870|
+---------+------+



In [6]:
print("no salting")
start = time.time()

naive = df.groupBy("campaign_id", "event_type", "device_type") \
    .agg(
        F.count("event_id").alias("total_events"),
        F.round(F.sum("revenue_usd"), 4).alias("total_revenue")
    )

naive.orderBy(F.col("total_events").desc()).show(5, truncate=False)
print(f"Naive time: {time.time() - start:.2f}s")

no salting


+-----------+----------+-----------+------------+-------------+
|campaign_id|event_type|device_type|total_events|total_revenue|
+-----------+----------+-----------+------------+-------------+
|camp_001   |impression|mobile     |1647521     |9059.5324    |
|camp_003   |impression|mobile     |1646563     |9053.8181    |
|camp_002   |impression|mobile     |1646398     |9053.3026    |
|camp_001   |impression|desktop    |760473      |4183.3991    |
|camp_003   |impression|desktop    |758979      |4175.5575    |
+-----------+----------+-----------+------------+-------------+
only showing top 5 rows
Naive time: 2.57s


In [5]:
print("SALTED")
start = time.time()

SALT_BUCKETS = 10

# Stage 1 — add random salt, aggregate on salted key
# salt is random per row so camp_001 spreads across 10 buckets
stage1 = df \
    .withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int")) \
    .withColumn("salted_key",
        F.concat_ws("_", F.col("campaign_id"),
                         F.col("event_type"),
                         F.col("device_type"),
                         F.col("salt").cast("string"))
    ) \
    .groupBy("salted_key", "campaign_id", "event_type", "device_type") \
    .agg(
        F.count("event_id").alias("partial_events"),
        F.sum("revenue_usd").alias("partial_revenue")
    )

# Stage 2 — remove salt, sum the partial results
salted = stage1 \
    .groupBy("campaign_id", "event_type", "device_type") \
    .agg(
        F.sum("partial_events").alias("total_events"),
        F.round(F.sum("partial_revenue"), 4).alias("total_revenue")
    )

salted.orderBy(F.col("total_events").desc()).show(5, truncate=False)
print(f"Salted time: {time.time() - start:.2f}s")

SALTED


+-----------+----------+-----------+------------+-------------+
|campaign_id|event_type|device_type|total_events|total_revenue|
+-----------+----------+-----------+------------+-------------+
|camp_001   |impression|mobile     |1647521     |9059.5324    |
|camp_003   |impression|mobile     |1646563     |9053.8181    |
|camp_002   |impression|mobile     |1646398     |9053.3026    |
|camp_001   |impression|desktop    |760473      |4183.3991    |
|camp_003   |impression|desktop    |758979      |4175.5575    |
+-----------+----------+-----------+------------+-------------+
only showing top 5 rows
Salted time: 5.02s
